# Document Corpus Exploration
Sprint 0 output: understanding the final corpus dimensions and structural limitations before diving into NLP models.

**Key Highlights:**
- Uses `data/metadata/document_corpus.csv`.
- Exactly 322 total documents.

In [27]:
import pandas as pd

df = pd.read_csv('../data/metadata/document_corpus.csv')
print(f"Final Corpus Shape: {df.shape[0]} documents, {df.shape[1]} columns.")
display(df.columns.tolist())

Final Corpus Shape: 322 documents, 17 columns.


['doc_id',
 'source',
 'title',
 'url',
 'ministry',
 'category',
 'filename',
 'date',
 'date_precision',
 'doc_type',
 'tags',
 'extracted_text',
 'extracted_text_length',
 'rtve_summary',
 'ocr_quality_score',
 'flag_illegible',
 'analysis_text']

### Null Value Verification

In [28]:
df.isnull().sum()

doc_id                     0
source                     0
title                      0
url                        0
ministry                 167
category                 167
filename                 167
date                      81
date_precision             0
doc_type                   0
tags                     155
extracted_text           169
extracted_text_length      0
rtve_summary             155
ocr_quality_score          0
flag_illegible             0
analysis_text             16
dtype: int64

### Metadata Distributions

In [29]:
print("\n--- Date Precision ---")
print(df['date_precision'].value_counts(dropna=False))

print("\n--- Document Type ---")
print(df['doc_type'].value_counts(dropna=False))


--- Date Precision ---
date_precision
day        212
unknown     79
month       27
year         4
Name: count, dtype: int64

--- Document Type ---
doc_type
Resumen de Juicio (RTVE)    167
Intelligence note            68
Otro                         48
Restricted                   12
Secret                       10
Report                        8
Telephone transcript          5
Telex                         3
Manuscrito                    1
Name: count, dtype: int64


### Text Feasibility (`analysis_text`)
**Two major limitations to be aware of:**

1. **RTVE Texts**: Note that RTVE data (`R0...`) does not include native PDF extraction. The system strategically relies on the `rtve_summary` string as the master content mapped into the `analysis_text` column.
2. **Garbage OCR**: 16 specific documents from Moncloa generated severely corrupted characters due to being illegible handwriting. These have been specifically tagged with `flag_illegible = True`, resulting in a natively excluded `analysis_text` (`NaN`) so they won't accidentally poison our upcoming Unsupervised NLP topics or mappings.

In [30]:
# Notice that exactly 16 documents have null analysis_text
print(f"Illegible / excluded documents: {df['flag_illegible'].sum()}")
print(f"Null analysis_text count: {df['analysis_text'].isnull().sum()}")

# Length distribution of the actual valid text strings we'll pipe into embeddings
valid_text_lengths = df['analysis_text'].dropna().apply(lambda x: len(str(x)))

print("\n--- 'analysis_text' Length Stats ---")
print(valid_text_lengths.describe())

Illegible / excluded documents: 16
Null analysis_text count: 16

--- 'analysis_text' Length Stats ---
count      306.000000
mean      4063.539216
std       9498.868148
min         53.000000
25%        303.000000
50%        303.000000
75%       3603.750000
max      85531.000000
Name: analysis_text, dtype: float64


In [31]:
import pandas as pd
df = pd.read_csv('../data/metadata/rtve_documents.csv')
df.head()

,name,pages,size_kb,summary,tags,link
0,Vista oral 2/81 del Consejo Supremo de Justici...,3,-,El juicio oral 2/81 celebrado en febrero de 19...,C/SG/2820/20-02-82 | DTOR. | Vista oral 2/81,https://23fbuscador.rtve.es/document/ocr/1860
1,Vista oral 2/81 del Consejo Supremo de Justici...,4,-,Resumen global del documento:\n\nEl documento ...,C/SG/2896/22-02-82 | Vista oral 2/81 | Consejo...,https://23fbuscador.rtve.es/document/ocr/1859
2,Vista oral 2/81 del Consejo Supremo de Justici...,5,-,Resumen global del documento:\n\nEl documento ...,C/SG/2992/24-02-82 | Vista Oral 2/81 | Consejo...,https://23fbuscador.rtve.es/document/ocr/1858
3,Vista oral 2/81 del Consejo Supremo de Justici...,6,-,El documento recoge el desarrollo de la sesión...,C/SG/3.081/25-02-82 | Vista Oral 2/81 | Consej...,https://23fbuscador.rtve.es/document/ocr/1857
4,Vista oral 2/81 del Consejo Supremo de Justici...,6,-,Resumen global del documento sobre la sesión d...,C/SG/3.249/26-02-82 | SG | Consejo Supremo de ...,https://23fbuscador.rtve.es/document/ocr/1856


In [32]:
df = pd.read_csv('../data/metadata/document_corpus.csv')
moncloa_df = df[df['source']== 'Moncloa']
rtve_df = df[df['source'] == 'RTVE']

In [33]:
rtve_df

,doc_id,source,title,url,ministry,category,filename,date,date_precision,doc_type,tags,extracted_text,extracted_text_length,rtve_summary,ocr_quality_score,flag_illegible,analysis_text
155,R001,RTVE,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1860,NaN,NaN,NaN,1982-02-01,month,Resumen de Juicio (RTVE),C/SG/2820/20-02-82 | DTOR. | Vista oral 2/81,NaN,0,El juicio oral 2/81 celebrado en febrero de 19...,0.0,False,El juicio oral 2/81 celebrado en febrero de 19...
156,R002,RTVE,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1859,NaN,NaN,NaN,1982-02-22,day,Resumen de Juicio (RTVE),C/SG/2896/22-02-82 | Vista oral 2/81 | Consejo...,NaN,0,Resumen global del documento:\n\nEl documento ...,0.0,False,Resumen global del documento:\n\nEl documento ...
157,R003,RTVE,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1858,NaN,NaN,NaN,1982-02-01,month,Resumen de Juicio (RTVE),C/SG/2992/24-02-82 | Vista Oral 2/81 | Consejo...,NaN,0,Resumen global del documento:\n\nEl documento ...,0.0,False,Resumen global del documento:\n\nEl documento ...
158,R004,RTVE,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1857,NaN,NaN,NaN,1982-02-01,month,Resumen de Juicio (RTVE),C/SG/3.081/25-02-82 | Vista Oral 2/81 | Consej...,NaN,0,El documento recoge el desarrollo de la sesión...,0.0,False,El documento recoge el desarrollo de la sesión...
159,R005,RTVE,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1856,NaN,NaN,NaN,1982-02-26,day,Resumen de Juicio (RTVE),C/SG/3.249/26-02-82 | SG | Consejo Supremo de ...,NaN,0,Resumen global del documento sobre la sesión d...,0.0,False,Resumen global del documento sobre la sesión d...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,R163,RTVE,Documento manuscrito de posible planificación ...,https://23fbuscador.rtve.es/document/ocr/1698,NaN,NaN,NaN,NaN,unknown,Resumen de Juicio (RTVE),Junta Superior | E.M. | U.S. Operativas,NaN,0,Resumen global del documento:\n\nEl documento ...,0.0,False,Resumen global del documento:\n\nEl documento ...
318,R164,RTVE,"""Documentación con una presunta planificación ...",https://23fbuscador.rtve.es/document/ocr/1697,NaN,NaN,NaN,1980-11-01,month,Resumen de Juicio (RTVE),NOVIEMBRE 1980 | Juez de Instrucción | Alcalde...,NaN,0,El documento de noviembre de 1980 y 1981 prese...,0.0,False,El documento de noviembre de 1980 y 1981 prese...
319,R165,RTVE,Conversaciones telefónicas de (presuntamente) ...,https://23fbuscador.rtve.es/document/ocr/1696,NaN,NaN,NaN,1981-02-24,day,Resumen de Juicio (RTVE),"UNIDAD MILITAR ""PRADO"" | Cuartel de Retamares ...",NaN,0,El documento recopila registros de llamadas te...,0.0,False,El documento recopila registros de llamadas te...
320,R166,RTVE,Transcripción de conversación telefónica de Ga...,https://23fbuscador.rtve.es/document/ocr/1695,NaN,NaN,NaN,NaN,unknown,Resumen de Juicio (RTVE),Juan | Villaviciosa y Pavía | Guardias Civiles...,NaN,0,El documento recoge una conversación centrada ...,0.0,False,El documento recoge una conversación centrada ...


In [34]:
moncloa_df

,doc_id,source,title,url,ministry,category,filename,date,date_precision,doc_type,tags,extracted_text,extracted_text_length,rtve_summary,ocr_quality_score,flag_illegible,analysis_text
0,M001,Moncloa,Transcripción de conversación telefónica de (p...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,Guardia Civil,23F_1._Conversacion_telefonica_GARCIA_CARRES_y...,NaN,unknown,Telephone transcript,NaN,--- Página 1 ---\n4\n\n0303730\no\n\nCONVERSAC...,6317,NaN,0.620069,False,--- Página 1 ---\n4\n\n0303730\no\n\nCONVERSAC...
1,M002,Moncloa,Transcripción de conversación telefónica de Ga...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,Guardia Civil,23F_2._Conversacion_telefonica_GARCIA_CARRES.pdf,NaN,unknown,Telephone transcript,NaN,--- Página 1 ---\nCONVERSACION ENTRE GARCIA CA...,2119,NaN,0.587940,False,--- Página 1 ---\nCONVERSACION ENTRE GARCIA CA...
2,M003,Moncloa,Conversaciones telefónicas de (presuntamente) ...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,Guardia Civil,23F_3._Conversaciones_telefonicas_unidad_milit...,1981-02-23,day,Telephone transcript,NaN,"--- Página 1 ---\n3,30h. de la madrugeda del d...",14482,NaN,0.650598,False,"--- Página 1 ---\n3,30h. de la madrugeda del d..."
3,M004,Moncloa,Documentación con una presunta planificación d...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,Guardia Civil,23F_4._Documento_planificacion_del_golpe.pdf,NaN,unknown,Otro,NaN,"--- Página 1 ---\n, PE ido HA a\nVen crusriana...",10054,NaN,0.576177,False,"--- Página 1 ---\n, PE ido HA a\nVen crusriana..."
4,M005,Moncloa,Documento manuscrito de posible planificación ...,https://www.lamoncloa.gob.es/consejodeministro...,Interior,Guardia Civil,23F_5._Documento_manuscrito_planificacion_del_...,NaN,unknown,Manuscrito,NaN,--- Página 1 ---\no o A\no £ y A\nad rom pp yo...,993,NaN,0.431694,False,--- Página 1 ---\no o A\no £ y A\nad rom pp yo...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150,M151,Moncloa,D.27._AGA-83-09301_exp._5,https://www.lamoncloa.gob.es/consejodeministro...,Exteriores,Exteriores,D.27._AGA-83-09301_exp._5.pdf,1981-03-19,day,Otro,NaN,--- Página 1 ---\nDE Comunicado pi ...DIRECCIO...,794,NaN,0.592308,False,--- Página 1 ---\nDE Comunicado pi ...DIRECCIO...
151,M152,Moncloa,D.28._AGA-83-09301_exp._5,https://www.lamoncloa.gob.es/consejodeministro...,Exteriores,Exteriores,D.28._AGA-83-09301_exp._5.pdf,NaN,day,Otro,NaN,--- Página 1 ---\n5/81\n\nMENSAJE DE SU MAJEST...,1376,NaN,0.683962,False,--- Página 1 ---\n5/81\n\nMENSAJE DE SU MAJEST...
152,M153,Moncloa,D.29._AGA-83-09301_exp._5,https://www.lamoncloa.gob.es/consejodeministro...,Exteriores,Exteriores,D.29._AGA-83-09301_exp._5.pdf,1981-02-25,day,Otro,NaN,--- Página 1 ---\nY\nE\nSEVABA Coma as LE - |\...,2078,NaN,0.717868,False,--- Página 1 ---\nY\nE\nSEVABA Coma as LE - |\...
153,M154,Moncloa,D.30._AGA-83-09301_exp._5,https://www.lamoncloa.gob.es/consejodeministro...,Exteriores,Exteriores,D.30._AGA-83-09301_exp._5.pdf,1981-02-25,day,Otro,NaN,--- Página 1 ---\nLp ESE D 116 /- Yes\n\nA\n\n...,1508,NaN,0.794393,False,--- Página 1 ---\nLp ESE D 116 /- Yes\n\nA\n\n...


In [35]:
import pandas as pd
df = pd.read_csv('../data/metadata/document_corpus.csv')

# Ver fechas extraídas
print(df['date'].value_counts(dropna=False).head(20))

# Ver qué más extrajo el LLM
print(df.columns.tolist())

# Muestra de los primeros docs con fecha
print(df[df['date'].notna()][['doc_id','source','date','doc_type']].head(20))

date
NaN           81
1981-02-23    30
1981-02-24     8
1981-02-01     7
1981-02-25     5
1981-01-01     5
1981-02-26     4
1982-04-28     4
1982-03-16     4
1982-04-01     4
1981-03-19     3
1982-04-19     3
1982-03-05     3
1981-02-03     3
1981-04-22     3
1981-03-25     3
1982-02-01     3
1982-03-01     3
1981-05-03     3
1981-03-01     3
Name: count, dtype: int64
['doc_id', 'source', 'title', 'url', 'ministry', 'category', 'filename', 'date', 'date_precision', 'doc_type', 'tags', 'extracted_text', 'extracted_text_length', 'rtve_summary', 'ocr_quality_score', 'flag_illegible', 'analysis_text']
   doc_id   source        date              doc_type
2    M003  Moncloa  1981-02-23  Telephone transcript
9    M010  Moncloa  1981-11-16     Intelligence note
10   M011  Moncloa  1981-10-10     Intelligence note
11   M012  Moncloa  1981-02-24                Report
12   M013  Moncloa  1981-02-25                Report
13   M014  Moncloa  1981-02-26                Report
14   M015  Moncloa  1981

In [36]:
print(df[df['date'].isna()][['doc_id','source','doc_type','extracted_text_length']].to_string())

    doc_id   source                  doc_type  extracted_text_length
0     M001  Moncloa      Telephone transcript                   6317
1     M002  Moncloa      Telephone transcript                   2119
3     M004  Moncloa                      Otro                  10054
4     M005  Moncloa                Manuscrito                    993
5     M006  Moncloa      Telephone transcript                      0
6     M007  Moncloa         Intelligence note                   7305
7     M008  Moncloa                     Telex                  63216
8     M009  Moncloa                    Report                   2177
19    M020  Moncloa         Intelligence note                      0
22    M023  Moncloa                    Secret                  11224
24    M025  Moncloa         Intelligence note                  24867
25    M026  Moncloa         Intelligence note                   2037
29    M030  Moncloa                      Otro                    676
30    M031  Moncloa               

In [37]:
import pandas as pd
df = pd.read_csv('../data/metadata/document_corpus.csv')

print("=== SHAPE ===")
print(df.shape)

print("\n=== COLUMNS ===")
print(df.columns.tolist())

print("\n=== NULLS ===")
print(df.isnull().sum())

print("\n=== DATE FORMAT SAMPLE ===")
print(df[df['date'].notna()][['doc_id','date','date_precision']].head(10))

print("\n=== DATE PRECISION DISTRIBUTION ===")
print(df['date_precision'].value_counts(dropna=False))

print("\n=== DOC_TYPE DISTRIBUTION ===")
print(df['doc_type'].value_counts(dropna=False))

print("\n=== SOURCE DISTRIBUTION ===")
print(df['source'].value_counts())

print("\n=== ANALYSIS_TEXT NULL COUNT ===")
print(df['analysis_text'].isna().sum())

print("\n=== SAMPLE ANALYSIS_TEXT LENGTH ===")
print(df[df['analysis_text'].notna()]['analysis_text'].str.len().describe())

=== SHAPE ===
(322, 17)

=== COLUMNS ===
['doc_id', 'source', 'title', 'url', 'ministry', 'category', 'filename', 'date', 'date_precision', 'doc_type', 'tags', 'extracted_text', 'extracted_text_length', 'rtve_summary', 'ocr_quality_score', 'flag_illegible', 'analysis_text']

=== NULLS ===
doc_id                     0
source                     0
title                      0
url                        0
ministry                 167
category                 167
filename                 167
date                      81
date_precision             0
doc_type                   0
tags                     155
extracted_text           169
extracted_text_length      0
rtve_summary             155
ocr_quality_score          0
flag_illegible             0
analysis_text             16
dtype: int64

=== DATE FORMAT SAMPLE ===
   doc_id        date date_precision
2    M003  1981-02-23            day
9    M010  1981-11-16            day
10   M011  1981-10-10            day
11   M012  1981-02-24       